# Customer Churn Prediction - Model Training

In [ ]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.metrics import (
    accuracy_score, precision_score,
    recall_score, f1_score, roc_auc_score
)


## Load Data Splits

In [ ]:
# Load the SMOTE-balanced splits saved by Train-Test Split.ipynb
X_train = pd.read_csv('../data/splits/X_train.csv')
y_train = pd.read_csv('../data/splits/y_train.csv').squeeze()
X_test  = pd.read_csv('../data/splits/X_test.csv')
y_test  = pd.read_csv('../data/splits/y_test.csv').squeeze()

print('X_train shape :', X_train.shape)
print('X_test  shape :', X_test.shape)
print('y_train value counts:')
print(y_train.value_counts())
print('y_test  value counts:')
print(y_test.value_counts())


## 1. Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)
print('Logistic Regression trained successfully')


## 2. Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(random_state=42, max_depth=6)
dt.fit(X_train, y_train)

y_pred_dt = dt.predict(X_test)
print('Decision Tree trained successfully')


## 3. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=10)
rf.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
print('Random Forest trained successfully')


## 4. XGBoost

In [ ]:
from xgboost import XGBClassifier

xgb = XGBClassifier(n_estimators=100, random_state=42, max_depth=5, eval_metric='logloss')
xgb.fit(X_train, y_train)

y_pred_xgb = xgb.predict(X_test)
print('XGBoost trained successfully')


## 5. Compare Model Performance

In [ ]:
# Helper: collect all metrics for one model
def get_metrics(name, y_true, y_pred, y_proba):
    return {
        'Model'     : name,
        'Accuracy'  : round(accuracy_score(y_true, y_pred), 4),
        'Precision' : round(precision_score(y_true, y_pred), 4),
        'Recall'    : round(recall_score(y_true, y_pred), 4),
        'F1 Score'  : round(f1_score(y_true, y_pred), 4),
        'ROC-AUC'   : round(roc_auc_score(y_true, y_proba), 4),
    }

results = [
    get_metrics('Logistic Regression', y_test, y_pred_lr,  lr.predict_proba(X_test)[:, 1]),
    get_metrics('Decision Tree',       y_test, y_pred_dt,  dt.predict_proba(X_test)[:, 1]),
    get_metrics('Random Forest',       y_test, y_pred_rf,  rf.predict_proba(X_test)[:, 1]),
    get_metrics('XGBoost',             y_test, y_pred_xgb, xgb.predict_proba(X_test)[:, 1]),
]

comparison_df = pd.DataFrame(results)
print(comparison_df.to_string(index=False))


## 6. Hyperparameter Tuning (XGBoost with GridSearchCV)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth'     : [3, 5, 7],
    'n_estimators'  : [100, 200],
    'learning_rate' : [0.01, 0.1, 0.2]
}

grid_search = GridSearchCV(
    XGBClassifier(random_state=42, eval_metric='logloss'),
    param_grid,
    scoring='recall',
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print('Best Parameters :', grid_search.best_params_)
print('Best Recall (CV):', round(grid_search.best_score_, 4))


### Evaluate Tuned XGBoost vs Base XGBoost

In [ ]:
# Get the best model from grid search
best_xgb = grid_search.best_estimator_

y_pred_best_xgb = best_xgb.predict(X_test)

tuned_metrics = get_metrics(
    'XGBoost (Tuned)',
    y_test,
    y_pred_best_xgb,
    best_xgb.predict_proba(X_test)[:, 1]
)

# Show before vs after tuning
base_metrics = get_metrics(
    'XGBoost (Base)',
    y_test,
    y_pred_xgb,
    xgb.predict_proba(X_test)[:, 1]
)

compare_tuning = pd.DataFrame([base_metrics, tuned_metrics])
print(compare_tuning.to_string(index=False))


## 7. Save the Best Model

In [ ]:
os.makedirs('../models', exist_ok=True)

# Save all 4 models
joblib.dump(lr,        '../models/logistic_regression.pkl')
joblib.dump(dt,        '../models/decision_tree.pkl')
joblib.dump(rf,        '../models/random_forest.pkl')
joblib.dump(best_xgb,  '../models/xgboost_tuned.pkl')

print('All models saved to /models folder:')
print('  logistic_regression.pkl')
print('  decision_tree.pkl')
print('  random_forest.pkl')
print('  xgboost_tuned.pkl')


## 8. Churn Probability Prediction on Sample Customers

In [ ]:
# Load saved model and predict on 5 sample customers
model = joblib.load('../models/xgboost_tuned.pkl')

sample = X_test.head(5).copy()
sample['Actual Churn']      = y_test.values[:5]
sample['Churn Probability'] = model.predict_proba(X_test.head(5))[:, 1].round(4)
sample['Prediction']        = model.predict(X_test.head(5))

print('=== Churn Probability Prediction (XGBoost Tuned) ===')
print(sample[['Actual Churn', 'Churn Probability', 'Prediction']].to_string())
print('0 = Will Stay   |   1 = Will Churn')
